# NB_Silver_To_Gold
Enterprise Banking Data Platform

Author: Rakesh Soma

Transforms Silver layer into Gold dimensional model for Power BI.

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime

spark = SparkSession.builder.appName("EnterpriseBankingFabric-Gold").getOrCreate()

print("="*70)
print("Gold Layer Started")
print(datetime.now())
print("="*70)

# Enable Delta optimizations
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled","true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled","true")

# -----------------------------
# Load Silver Tables
# -----------------------------
customer = spark.table("Silver.CustomerMaster")
branch   = spark.table("Silver.BranchMaster")
currency = spark.table("Silver.CurrencyMaster")
account  = spark.table("Silver.AccountMaster")
loan     = spark.table("Silver.LoanMaster")
card     = spark.table("Silver.CardMaster")
txn      = spark.table("Silver.Transaction")

# -----------------------------
# Build Dimensions
# -----------------------------
DimCustomer = customer.select(
    "CustomerID",
    "CustomerName",
    "CustomerType",
    "RiskCategory",
    "CountryID",
    "LoadDate"
).dropDuplicates()

DimBranch = branch.select(
    "BranchID",
    "BranchCode",
    "BranchName",
    "CountryID",
    "LoadDate"
).dropDuplicates()

DimCurrency = currency.select(
    "CurrencyID",
    "CurrencyCode",
    "CurrencyName",
    "Symbol"
).dropDuplicates()

# -----------------------------
# Build Facts
# -----------------------------
FactTransaction = txn.select(
    "TransactionID",
    "CustomerID",
    "AccountID",
    "BranchID",
    "CurrencyID",
    "TransactionAmount",
    "TransactionDate"
)

FactLoan = loan.select(
    "LoanID",
    "CustomerID",
    "BranchID",
    "LoanAmount",
    "InterestRate",
    "LoanStatus"
)

FactCard = card.select(
    "CardID",
    "CustomerID",
    "BranchID",
    "CardType",
    "CreditLimit"
)

# -----------------------------
# Banking KPIs
# -----------------------------
TotalCustomers = DimCustomer.count()
TotalTransactions = FactTransaction.count()
TotalLoanAmount = FactLoan.agg(sum("LoanAmount")).first()[0]
TotalCards = FactCard.count()

print(f"Customers        : {TotalCustomers}")
print(f"Transactions     : {TotalTransactions}")
print(f"Loan Amount      : {TotalLoanAmount}")
print(f"Cards            : {TotalCards}")

# -----------------------------
# Save Gold Tables
# -----------------------------
DimCustomer.write.mode("overwrite").format("delta").saveAsTable("Gold.DimCustomer")
DimBranch.write.mode("overwrite").format("delta").saveAsTable("Gold.DimBranch")
DimCurrency.write.mode("overwrite").format("delta").saveAsTable("Gold.DimCurrency")

FactTransaction.write.mode("overwrite").format("delta").saveAsTable("Gold.FactTransaction")
FactLoan.write.mode("overwrite").format("delta").saveAsTable("Gold.FactLoan")
FactCard.write.mode("overwrite").format("delta").saveAsTable("Gold.FactCard")

# -----------------------------
# Optimize Gold Tables
# -----------------------------
gold_tables=[
"DimCustomer",
"DimBranch",
"DimCurrency",
"FactTransaction",
"FactLoan",
"FactCard"
]

for t in gold_tables:
    try:
        spark.sql(f"OPTIMIZE Gold.{t}")
        print(f"Optimized {t}")
    except:
        pass

print("="*70)
print("Gold Layer Completed Successfully")
print(datetime.now())
print("="*70)
